In [1]:
# ============================================================
# CELDA 1: Instalación de dependencias
# ============================================================
!pip install langchain langchain-community chromadb sentence-transformers pypdf tiktoken -q


In [3]:
pip install langchain-core



Note: you may need to restart the kernel to use updated packages.


In [19]:
from pathlib import Path

base = Path("/Users/migueltorres/Documents/Documents - Miguel’s MacBook Pro/")
print([p.name for p in base.iterdir()])




['ENABLE', 'Nissan Sentra', 'Corporate Finance', 'Programa instalador Adobe', 'RECIBOS DE NOMINA', 'Screenshot', '.DS_Store', 'Investment', 'MATLAB Investments', 'Declaraciones Anuales ', 'KPMG Docs.zip', 'Documentos Mercedes AMG A35', 'declaraciones SAT', '.localized', 'Renta Naranjo Docs.zip', 'Clases de inglés KTSA', 'Pagina de internet', 'RECIBOS DE NOMINA.zip', 'Renta Naranjo Docs', '~$P Sales Cloud certification questions.docx', 'Documentos de Matlab', "CV's 2022", 'Documentos CIVIC 2024', 'WindowsSupport', 'Certificacioens Docs', 'Buró de Crédito', 'Photo Cases', 'recibosdenminamigueltorres Enable', 'Financial Markets', 'GBM Docs', 'EMP - Docs', 'Formato de Oficinas Miguel Torres.pdf', 'Diplomado AI and LLM for Financial Markets', 'KPMG Docs', 'Adobe', 'Documentos Maestría', 'China trip', 'macbook pro 2014', '~$ANALISIS PORTAFOLIOS.xlsx', 'Go pro', 'Diplomado AI and LLM for FM', 'Documentos personales de identificacion', 'MATLAB', 'Fotns', 'Comprobantes de pago tec', 'INE Mi

In [2]:
# ============================================================
# CELDA 2: Carga y chunking de documentos
# ============================================================
import json
from langchain_core.documents import Document
from pathlib import Path

documents = []


CHUNKS_PATH = Path("/Users/migueltorres/Documents/Documents - Miguel’s MacBook Pro/Diplomado AI and LLM for FM/Modulo 3/Proyecto Final/sec_data/chunks/all_chunks.jsonl")


with open(CHUNKS_PATH, "r") as f:
    for line in f:
        item = json.loads(line)
        documents.append(Document(
            page_content=item["text"],
            metadata={
                "source": item.get("source", "unknown"),
                "section": item.get("section", "unknown"),
                "filing_type": item.get("filing_type", "unknown")
            }
        ))

print(f"Total chunks cargados: {len(documents)}")



Total chunks cargados: 11940


In [3]:
!pip install chromadb sentence-transformers rank_bm25 -q


In [4]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=documents,  # tu lista de Documents cargados
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

print(f"✅ {vectorstore._collection.count()} documentos indexados en ChromaDB")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ 11940 documentos indexados en ChromaDB


In [5]:
query = "What are the company's total revenues?"
results = vectorstore.similarity_search(query, k=5)

for i, doc in enumerate(results):
    print(f"\n--- Resultado {i+1} ---")
    print(doc.page_content[:300])
    print("Metadata:", doc.metadata)



--- Resultado 1 ---
94,907
 
 
$
96,216
 
 
International
28,424
 
 
30,169
 
 
Total long-lived assets
$
123,331
 
 
$
126,385
 
 
For revenues by geography, see Note 2.
 
30
Table of Contents
Alphabet Inc.
ITEM 2.
MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS
Please read the fo
Metadata: {'filing_type': '10-Q', 'source': 'unknown', 'section': 'unknown'}

--- Resultado 2 ---
B shares authorized, 
351
 million and 
367
 million shares issued and outstanding, as of March 31, 2023 and December 31, 2022, respectively
—
 
—
 
Additional paid-in capital
66,535
 
64,444
 
Accumulated other comprehensive loss
(2,981)
(3,530)
Retained earnings
61,241
 
64,799
 
Total stockholder
Metadata: {'section': 'unknown', 'source': 'unknown', 'filing_type': '10-Q'}

--- Resultado 3 ---
$
921,983
 
See accompanying notes.
4
Table of Contents
Alphabet Inc.
Alphabet Inc.
CONSOLIDATED STATEMENTS OF INCOME
(in millions, except per share amounts; unaudited)
Three Months

In [6]:
eval_queries = [
    {"query": "What are the total revenues?", "expected_source": "10-K"},
    {"query": "What are the risk factors?", "expected_source": "10-K"},
    # agrega más según tus datos
]


In [7]:
# Ver qué campos de metadata tienes
sample = documents[0].metadata
print("Campos disponibles:", sample.keys())
print("Ejemplo:", sample)


Campos disponibles: dict_keys(['source', 'section', 'filing_type'])
Ejemplo: {'source': 'unknown', 'section': 'unknown', 'filing_type': '10-K'}


In [8]:
# --- Metadata Filtering: filtrado por tipo de filing ---

query = "What are the company's main risk factors?"

# Sin filtro (baseline)
results_no_filter = vectorstore.similarity_search(query, k=5)

# Filtrado solo 10-K (reportes anuales, más completos para risk factors)
results_10k = vectorstore.similarity_search(query, k=5, filter={"filing_type": "10-K"})

# Filtrado solo 10-Q (reportes trimestrales)
results_10q = vectorstore.similarity_search(query, k=5, filter={"filing_type": "10-Q"})

print("=" * 60)
print("🔍 BASELINE (sin filtro)")
print("=" * 60)
for i, doc in enumerate(results_no_filter):
    print(f"\n--- Resultado {i+1} | filing_type: {doc.metadata.get('filing_type')} ---")
    print(doc.page_content[:200])

print("\n" + "=" * 60)
print("📄 FILTRADO: solo 10-K")
print("=" * 60)
for i, doc in enumerate(results_10k):
    print(f"\n--- Resultado {i+1} ---")
    print(doc.page_content[:200])

print("\n" + "=" * 60)
print("📄 FILTRADO: solo 10-Q")
print("=" * 60)
for i, doc in enumerate(results_10q):
    print(f"\n--- Resultado {i+1} ---")
    print(doc.page_content[:200])



🔍 BASELINE (sin filtro)

--- Resultado 1 | filing_type: 10-Q ---
hat we currently believe are not material, may also become important factors that adversely affect our business. If any of the following risks actually occurs, our business, financial condition, resul

--- Resultado 2 | filing_type: 10-Q ---
ccurs, our business, financial condition, results of operations, and future prospects could be materially and adversely affected. In that event, the trading price of our Class A common stock could dec

--- Resultado 3 | filing_type: 10-Q ---
should consider carefully the risks and uncertainties described below, in addition to other information contained in this Quarterly Report on Form 10-Q, including our condensed consolidated financial 

--- Resultado 4 | filing_type: 10-Q ---
should consider carefully the risks and uncertainties described below, in addition to other information contained in this Quarterly Report on Form 10-Q, including our condensed consolidated financial 

--- Res

In [9]:
# --- Búsqueda Híbrida: Semántica (ChromaDB) + Léxica (BM25) ---

from rank_bm25 import BM25Okapi

# Preparar corpus para BM25 (tokenización simple)
corpus = [doc.page_content for doc in documents]
tokenized_corpus = [text.lower().split() for text in corpus]

bm25 = BM25Okapi(tokenized_corpus)

def hybrid_search(query, vectorstore, bm25, documents, k=5, alpha=0.5):
    """
    Combina búsqueda semántica y BM25.
    alpha=1.0 → solo semántica
    alpha=0.0 → solo BM25
    """
    # Semántica
    sem_results = vectorstore.similarity_search_with_score(query, k=len(documents))
    sem_scores = {}
    for doc, score in sem_results:
        sem_scores[doc.page_content[:100]] = 1 - score  # menor distancia = mejor

    # BM25
    tokenized_query = query.lower().split()
    bm25_scores_raw = bm25.get_scores(tokenized_query)

    # Normalizar ambos scores a [0, 1]
    sem_vals = list(sem_scores.values())
    sem_min, sem_max = min(sem_vals), max(sem_vals)
    bm25_min, bm25_max = min(bm25_scores_raw), max(bm25_scores_raw)

    hybrid_results = []
    for i, doc in enumerate(documents):
        key = doc.page_content[:100]
        
        # Normalizar
        s_sem = (sem_scores.get(key, 0) - sem_min) / (sem_max - sem_min + 1e-9)
        s_bm25 = (bm25_scores_raw[i] - bm25_min) / (bm25_max - bm25_min + 1e-9)
        
        # Score combinado
        combined = alpha * s_sem + (1 - alpha) * s_bm25
        hybrid_results.append((doc, combined))

    # Ordenar por score combinado (mayor = mejor)
    hybrid_results.sort(key=lambda x: x[1], reverse=True)
    return hybrid_results[:k]

# --- Ejecutar ---
query = "What are the company's main risk factors?"

results_hybrid = hybrid_search(query, vectorstore, bm25, documents, k=5, alpha=0.5)

print("=" * 60)
print("⚡ BÚSQUEDA HÍBRIDA (alpha=0.5)")
print("=" * 60)
for i, (doc, score) in enumerate(results_hybrid):
    print(f"\n--- Resultado {i+1} | Score: {score:.4f} | filing_type: {doc.metadata.get('filing_type')} ---")
    print(doc.page_content[:200])



⚡ BÚSQUEDA HÍBRIDA (alpha=0.5)

--- Resultado 1 | Score: 0.8197 | filing_type: 10-K ---
adversely affected, including higher costs or delays for our construction projects. We may take further actions as required by government authorities or others, or that we determine are in the best in

--- Resultado 2 | Score: 0.7868 | filing_type: 10-K ---
f it is probable that a loss has been incurred and the amount of the loss can be reasonably estimated. We evaluate developments that could affect prior disclosures or previously accrued liabilities, a

--- Resultado 3 | Score: 0.7690 | filing_type: 10-K ---
other relevant factors. We performed sensitivity analyses of significant assumptions to determine what changes in assumptions are particularly sensitive when assessing the likelihood of impairment, or

--- Resultado 4 | Score: 0.7410 | filing_type: 10-Q ---
putational Risks: If we pause or discontinue the design, development and manufacture of Intel 14A and future leading-edge nodes, it may gi

In [10]:
# --- Re-ranking: Cross-Encoder sobre resultados híbridos ---

from sentence_transformers import CrossEncoder

# Modelo ligero de re-ranking (funciona bien en CPU/Mac)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def hybrid_search_with_rerank(query, vectorstore, bm25, documents, k_candidates=20, k_final=5, alpha=0.5):
    """
    1. Búsqueda híbrida para obtener candidatos amplios
    2. Re-ranking con Cross-Encoder para refinar el top final
    """
    # Paso 1: Obtener candidatos con búsqueda híbrida
    candidates = hybrid_search(query, vectorstore, bm25, documents, k=k_candidates, alpha=alpha)
    
    # Paso 2: Preparar pares (query, passage) para el Cross-Encoder
    pairs = [[query, doc.page_content] for doc, _ in candidates]
    
    # Paso 3: Obtener scores del Cross-Encoder
    ce_scores = reranker.predict(pairs)
    
    # Paso 4: Combinar candidatos con sus nuevos scores
    reranked = list(zip([doc for doc, _ in candidates], ce_scores))
    reranked.sort(key=lambda x: x[1], reverse=True)
    
    return reranked[:k_final]

# --- Ejecutar ---
query = "What are the company's main risk factors?"

results_reranked = hybrid_search_with_rerank(
    query, vectorstore, bm25, documents,
    k_candidates=20,  # candidatos amplios del híbrido
    k_final=5,         # top final después del re-ranking
    alpha=0.5
)

print("=" * 60)
print("🏆 RE-RANKING (Cross-Encoder sobre Híbrido)")
print("=" * 60)
for i, (doc, score) in enumerate(results_reranked):
    print(f"\n--- Resultado {i+1} | CE Score: {score:.4f} | filing_type: {doc.metadata.get('filing_type')} ---")
    print(doc.page_content[:200])


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

🏆 RE-RANKING (Cross-Encoder sobre Híbrido)

--- Resultado 1 | CE Score: 1.8896 | filing_type: 10-Q ---
hat we currently believe are not material, may also become important factors that adversely affect our business. If any of the following risks actually occurs, our business, financial condition, resul

--- Resultado 2 | CE Score: -0.0364 | filing_type: 10-Q ---
should consider carefully the risks and uncertainties described below, in addition to other information contained in this Quarterly Report on Form 10-Q, including our condensed consolidated financial 

--- Resultado 3 | CE Score: -0.1767 | filing_type: 10-Q ---
should consider carefully the risks and uncertainties described below, in addition to other information contained in this Quarterly Report on Form 10-Q, including our condensed consolidated financial 

--- Resultado 4 | CE Score: -0.2166 | filing_type: 10-Q ---
ount of the loss can be reasonably estimated. We evaluate developments that could affect prior disclosures or p

In [11]:
# --- Tabla de Ablación: Comparativa de técnicas RAG ---
import pandas as pd

# Queries de evaluación con fragmentos relevantes esperados (ground truth parcial)
eval_queries = [
    {
        "query": "What are the company's main risk factors?",
        "relevant_keywords": ["risk", "factors", "uncertainties", "adverse"]
    },
    {
        "query": "What is the total revenue and net income?",
        "relevant_keywords": ["revenue", "net income", "total revenue", "earnings"]
    },
    {
        "query": "What are the significant accounting policies?",
        "relevant_keywords": ["accounting", "policies", "estimates", "financial statements"]
    },
    {
        "query": "Describe the company's debt and credit facilities",
        "relevant_keywords": ["debt", "credit", "borrowing", "loan", "facilities"]
    },
    {
        "query": "What is the business overview and strategy?",
        "relevant_keywords": ["business", "overview", "strategy", "operations", "segment"]
    },
]

def is_relevant(doc_text, keywords, threshold=2):
    """Un documento es 'relevante' si contiene al menos `threshold` keywords."""
    text_lower = doc_text.lower()
    hits = sum(1 for kw in keywords if kw.lower() in text_lower)
    return hits >= threshold

def evaluate_pipeline(name, search_func, queries, k=5):
    """Calcula Hit Rate y MRR para un pipeline dado."""
    hit_count = 0
    rr_sum = 0.0

    for q in queries:
        results = search_func(q["query"])

        # Extraer textos según formato de salida
        texts = []
        for item in results:
            if isinstance(item, tuple):
                doc, score = item
                texts.append(doc.page_content)
            else:
                texts.append(item.page_content)

        # Buscar primer resultado relevante
        found = False
        for rank, text in enumerate(texts[:k], start=1):
            if is_relevant(text, q["relevant_keywords"]):
                if not found:
                    rr_sum += 1.0 / rank
                    hit_count += 1
                    found = True
                break  # solo el primer hit cuenta para MRR

        # Si encontramos al menos uno relevante en top-k → hit
        if not found:
            for text in texts[:k]:
                if is_relevant(text, q["relevant_keywords"]):
                    hit_count += 1
                    # RR ya no se suma si el primero relevante no se encontró en orden
                    break

    n = len(queries)
    hit_rate = hit_count / n
    mrr = rr_sum / n
    return {"Técnica": name, "Hit Rate": f"{hit_rate:.2%}", "MRR": f"{mrr:.4f}"}


# --- Definir funciones de búsqueda para cada técnica ---

# 1. Baseline: búsqueda vectorial pura
def search_baseline(query):
    return vectorstore.similarity_search(query, k=5)

# 2. Metadata Filtering: solo 10-K
def search_metadata(query):
    return vectorstore.similarity_search(query, k=5, filter={"filing_type": "10-K"})

# 3. Híbrido (semántica + BM25)
def search_hybrid(query):
    return hybrid_search(query, vectorstore, bm25, documents, k=5, alpha=0.5)

# 4. Híbrido + Re-ranking
def search_rerank(query):
    return hybrid_search_with_rerank(query, vectorstore, bm25, documents,
                                     k_candidates=20, k_final=5, alpha=0.5)


# --- Ejecutar evaluación ---
pipelines = [
    ("1. Baseline (Vectorial)", search_baseline),
    ("2. Metadata Filter (10-K)", search_metadata),
    ("3. Híbrido (Sem + BM25)", search_hybrid),
    ("4. Híbrido + Re-ranking", search_rerank),
]

results_table = []
for name, func in pipelines:
    print(f"⏳ Evaluando: {name}...")
    metrics = evaluate_pipeline(name, func, eval_queries, k=5)
    results_table.append(metrics)
    print(f"   ✅ Hit Rate: {metrics['Hit Rate']} | MRR: {metrics['MRR']}")

# --- Mostrar tabla ---
print("\n" + "=" * 60)
print("📊 TABLA DE ABLACIÓN — Fase 6 RAG")
print("=" * 60)

df_ablation = pd.DataFrame(results_table)
print(df_ablation.to_string(index=False))

# Guardar como CSV
df_ablation.to_csv("ablation_rag_fase6.csv", index=False)
print("\n💾 Tabla guardada en: ablation_rag_fase6.csv")


⏳ Evaluando: 1. Baseline (Vectorial)...
   ✅ Hit Rate: 100.00% | MRR: 1.0000
⏳ Evaluando: 2. Metadata Filter (10-K)...
   ✅ Hit Rate: 100.00% | MRR: 0.8000
⏳ Evaluando: 3. Híbrido (Sem + BM25)...
   ✅ Hit Rate: 100.00% | MRR: 1.0000
⏳ Evaluando: 4. Híbrido + Re-ranking...
   ✅ Hit Rate: 100.00% | MRR: 1.0000

📊 TABLA DE ABLACIÓN — Fase 6 RAG
                  Técnica Hit Rate    MRR
  1. Baseline (Vectorial)  100.00% 1.0000
2. Metadata Filter (10-K)  100.00% 0.8000
  3. Híbrido (Sem + BM25)  100.00% 1.0000
  4. Híbrido + Re-ranking  100.00% 1.0000

💾 Tabla guardada en: ablation_rag_fase6.csv


In [12]:
# Queries técnicas diseñadas para diferenciar las técnicas
eval_queries_hard = [
    {
        "query": "What is the impact of ASC 842 lease accounting standard on financial statements?",
        "keywords": ["asc 842", "lease", "right-of-use", "operating lease", "liability"]
    },
    {
        "query": "How does the company evaluate goodwill impairment and what triggers testing?",
        "keywords": ["goodwill", "impairment", "fair value", "reporting unit", "intangible"]
    },
    {
        "query": "What are the cybersecurity risk factors and mitigation strategies?",
        "keywords": ["cybersecurity", "data breach", "security", "unauthorized access", "cyber"]
    },
    {
        "query": "What is the company's exposure to foreign currency exchange rate fluctuations?",
        "keywords": ["foreign currency", "exchange rate", "hedging", "forex", "translation"]
    },
    {
        "query": "How does the company recognize revenue across different product segments?",
        "keywords": ["revenue recognition", "asc 606", "performance obligation", "contract", "segment"]
    },
    {
        "query": "What are the material weaknesses identified in internal controls over financial reporting?",
        "keywords": ["material weakness", "internal control", "sarbanes", "deficiency", "remediation"]
    },
    {
        "query": "What is the company's debt maturity schedule and interest rate exposure?",
        "keywords": ["debt", "maturity", "interest rate", "long-term", "credit facility", "borrowing"]
    },
    {
        "query": "What environmental and climate-related risks does the company disclose?",
        "keywords": ["climate", "environmental", "sustainability", "emission", "regulation"]
    }
]


In [13]:
def is_relevant(doc_text, keywords, min_matches=3):
    """Un documento es relevante si contiene al menos min_matches keywords."""
    text_lower = doc_text.lower()
    matches = sum(1 for kw in keywords if kw.lower() in text_lower)
    return matches >= min_matches


In [14]:
# 1. BASELINE - búsqueda vectorial pura
def search_baseline(query, k=5):
    return vectorstore.similarity_search(query, k=k)

# 2. METADATA FILTER - solo 10-K
def search_metadata(query, k=5):
    return vectorstore.similarity_search(query, k=k, filter={"filing_type": "10-K"})

# HÍBRIDO MEJORADO - más peso semántico
def search_hybrid(query, k=5, alpha=0.7):  # Cambiado de 0.5 a 0.7
    sem_results = vectorstore.similarity_search_with_relevance_scores(query, k=20)
    
    from rank_bm25 import BM25Okapi
    corpus_tokens = [doc.page_content.lower().split() for doc in all_docs]
    bm25 = BM25Okapi(corpus_tokens)
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_top_idx = bm25_scores.argsort()[::-1][:20]
    bm25_results = [(all_docs[i], bm25_scores[i]) for i in bm25_top_idx]
    
    def normalize(scores):
        if not scores:
            return []
        min_s = min(s for _, s in scores)
        max_s = max(s for _, s in scores)
        if max_s == min_s:
            return [(doc, 1.0) for doc, _ in scores]
        return [(doc, (s - min_s) / (max_s - min_s)) for doc, s in scores]
    
    sem_norm = normalize(sem_results)
    bm25_norm = normalize(bm25_results)
    
    combined = {}
    for doc, score in sem_norm:
        combined[doc.page_content] = {"doc": doc, "score": alpha * score}
    for doc, score in bm25_norm:
        key = doc.page_content
        if key in combined:
            combined[key]["score"] += (1 - alpha) * score
        else:
            combined[key] = {"doc": doc, "score": (1 - alpha) * score}
    
    ranked = sorted(combined.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in ranked[:k]]

# RE-RANKING MEJORADO - pool más amplio (30 candidatos)
def search_reranking(query, k=5):
    candidates = search_hybrid(query, k=30)  # Ampliado de 20 a 30
    
    pairs = [[query, doc.page_content] for doc in candidates]
    scores = cross_encoder.predict(pairs)
    
    top_indices = scores.argsort()[::-1][:k]
    return [candidates[i] for i in top_indices]



In [15]:
# Necesario para BM25 en la búsqueda híbrida
all_docs = vectorstore.get(include=["documents", "metadatas"])

from langchain_core.documents import Document

all_docs = [
    Document(page_content=text, metadata=meta)
    for text, meta in zip(all_docs["documents"], all_docs["metadatas"])
]

print(f"📚 Total docs para BM25: {len(all_docs)}")


📚 Total docs para BM25: 11940


In [16]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ CrossEncoder cargado")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ CrossEncoder cargado


In [17]:
import pandas as pd

techniques = {
    "1. Baseline (Vector)": search_baseline,
    "2. Metadata (10-K)": search_metadata,
    "3. Hybrid (Sem+BM25)": search_hybrid,
    "4. Re-ranking (CE)": search_reranking
}

results_table = []

for tech_name, search_fn in techniques.items():
    hits = 0
    mrr_sum = 0.0
    
    for eval_item in eval_queries_hard:
        query = eval_item["query"]
        keywords = eval_item["keywords"]
        
        try:
            docs = search_fn(query, k=5)
        except Exception as e:
            print(f"⚠️ Error en '{tech_name}' con query: {query[:50]}... → {e}")
            docs = []
        
        # Hit Rate: ¿al menos 1 doc relevante en top-5?
        hit = False
        first_relevant_rank = None
        
        for rank, doc in enumerate(docs, start=1):
            if is_relevant(doc.page_content, keywords, min_matches=3):
                if not hit:
                    hit = True
                    first_relevant_rank = rank
                    break
        
        if hit:
            hits += 1
            mrr_sum += 1.0 / first_relevant_rank
    
    n = len(eval_queries_hard)
    hit_rate = hits / n
    mrr = mrr_sum / n
    
    results_table.append({
        "Technique": tech_name,
        "Hit Rate": f"{hit_rate:.2%}",
        "MRR": f"{mrr:.4f}",
        "Hits": f"{hits}/{n}"
    })
    print(f"✅ {tech_name}: Hit Rate={hit_rate:.2%}, MRR={mrr:.4f}")

# Tabla final
print("\n" + "="*60)
print("📊 ABLATION TABLE - RAG Pipeline SEC Edgar")
print("="*60)

df = pd.DataFrame(results_table)
print(df.to_string(index=False))


✅ 1. Baseline (Vector): Hit Rate=87.50%, MRR=0.5979
✅ 2. Metadata (10-K): Hit Rate=87.50%, MRR=0.7188
✅ 3. Hybrid (Sem+BM25): Hit Rate=75.00%, MRR=0.6562
✅ 4. Re-ranking (CE): Hit Rate=87.50%, MRR=0.5833

📊 ABLATION TABLE - RAG Pipeline SEC Edgar
           Technique Hit Rate    MRR Hits
1. Baseline (Vector)   87.50% 0.5979  7/8
  2. Metadata (10-K)   87.50% 0.7188  7/8
3. Hybrid (Sem+BM25)   75.00% 0.6562  6/8
  4. Re-ranking (CE)   87.50% 0.5833  7/8


In [18]:
print("🔍 Diagnóstico: query que falla en Hybrid pero no en Baseline\n")

for eval_item in eval_queries_hard:
    query = eval_item["query"]
    keywords = eval_item["keywords"]
    
    # Baseline
    docs_base = search_baseline(query, k=5)
    hit_base = any(is_relevant(d.page_content, keywords) for d in docs_base)
    
    # Hybrid
    docs_hybrid = search_hybrid(query, k=5)
    hit_hybrid = any(is_relevant(d.page_content, keywords) for d in docs_hybrid)
    
    status = "✅" if hit_base == hit_hybrid else "❌ DIVERGE"
    print(f"{status} | Base={hit_base} | Hybrid={hit_hybrid} | {query[:70]}...")


🔍 Diagnóstico: query que falla en Hybrid pero no en Baseline

✅ | Base=True | Hybrid=True | What is the impact of ASC 842 lease accounting standard on financial s...
✅ | Base=True | Hybrid=True | How does the company evaluate goodwill impairment and what triggers te...
✅ | Base=True | Hybrid=True | What are the cybersecurity risk factors and mitigation strategies?...
✅ | Base=True | Hybrid=True | What is the company's exposure to foreign currency exchange rate fluct...
❌ DIVERGE | Base=True | Hybrid=False | How does the company recognize revenue across different product segmen...
✅ | Base=False | Hybrid=False | What are the material weaknesses identified in internal controls over ...
✅ | Base=True | Hybrid=True | What is the company's debt maturity schedule and interest rate exposur...
✅ | Base=True | Hybrid=True | What environmental and climate-related risks does the company disclose...


In [21]:
# ¿Qué encuentra cada técnica para "material weaknesses"?
query_mw = "What are the material weaknesses identified in internal controls over financial reporting?"
keywords_mw = ["material weakness", "internal control", "deficiency", "remediation", "audit"]

docs_base = search_baseline(query_mw, k=5)
print("🔍 Baseline - top 5 para 'material weaknesses':\n")
for i, doc in enumerate(docs_base, 1):
    content_lower = doc.page_content.lower()
    matched = [kw for kw in keywords_mw if kw in content_lower]
    print(f"  Doc {i}: {len(matched)} keywords → {matched}")
    print(f"          {doc.page_content[:150]}...\n")


🔍 Baseline - top 5 para 'material weaknesses':

  Doc 1: 3 keywords → ['material weakness', 'internal control', 'audit']
          ecurities and Exchange Commission and the PCAOB.
We conducted our audit in accordance with the standards of the PCAOB. Those standards require that we...

  Doc 2: 3 keywords → ['material weakness', 'internal control', 'audit']
          ecurities and Exchange Commission and the PCAOB.
We conducted our audit in accordance with the standards of the PCAOB. Those standards require that we...

  Doc 3: 3 keywords → ['material weakness', 'internal control', 'audit']
          ecurities and Exchange Commission and the PCAOB.
We conducted our audit in accordance with the standards of the PCAOB. Those standards require that we...

  Doc 4: 3 keywords → ['material weakness', 'internal control', 'audit']
          curities and Exchange Commission and the PCAOB. 
We conducted our audit in accordance with the standards of the PCAOB. Those standards require that we...



In [22]:
# Copiar las funciones search_hybrid_v2 y search_reranking_v2 de la celda anterior
# y luego ejecutar la evaluación completa:

techniques_v2 = {
    "1. Baseline (Vector)": search_baseline,
    "2. Metadata (10-K)": search_metadata,
    "3. Hybrid v2 (Sem+BM25)": search_hybrid_v2,
    "4. Re-ranking v2 (CE)": search_reranking_v2
}

results_table = []

for tech_name, search_fn in techniques_v2.items():
    hits = 0
    mrr_sum = 0.0
    
    for eval_item in eval_queries_hard:
        query = eval_item["query"]
        keywords = eval_item["keywords"]
        
        try:
            docs = search_fn(query, k=5)
        except Exception as e:
            print(f"⚠️ Error: {tech_name} → {e}")
            docs = []
        
        for rank, doc in enumerate(docs, start=1):
            if is_relevant(doc.page_content, keywords, min_matches=3):
                hits += 1
                mrr_sum += 1.0 / rank
                break
    
    n = len(eval_queries_hard)
    hit_rate = hits / n
    mrr = mrr_sum / n
    
    results_table.append({
        "Technique": tech_name,
        "Hit Rate": f"{hit_rate:.2%}",
        "MRR": f"{mrr:.4f}",
        "Hits": f"{hits}/{n}"
    })
    print(f"✅ {tech_name}: Hit Rate={hit_rate:.2%}, MRR={mrr:.4f}")

print("\n" + "="*60)
print("📊 ABLATION TABLE v2 - RAG Pipeline SEC Edgar")
print("="*60)
df = pd.DataFrame(results_table)
print(df.to_string(index=False))


NameError: name 'search_hybrid_v2' is not defined

In [23]:
query_rev = "How does the company recognize revenue across different product segments?"
keywords_rev = ["revenue recognition", "disaggregat", "product segment", "contract", "ASC 606"]

# Semántica pura (lo que Baseline encuentra)
docs_base = search_baseline(query_rev, k=5)
print("🟢 BASELINE top-5:")
for i, doc in enumerate(docs_base, 1):
    matched = [kw for kw in keywords_rev if kw in doc.page_content.lower()]
    print(f"  Doc {i}: {len(matched)} kw → {matched} | {doc.page_content[:100]}...")

print("\n" + "-"*60)

# Hybrid v2
docs_hyb = search_hybrid_v2(query_rev, k=5)
print("\n🔴 HYBRID v2 top-5:")
for i, doc in enumerate(docs_hyb, 1):
    matched = [kw for kw in keywords_rev if kw in doc.page_content.lower()]
    print(f"  Doc {i}: {len(matched)} kw → {matched} | {doc.page_content[:100]}...")

# ¿El doc relevante del baseline está en el hybrid?
base_relevant = None
for doc in docs_base:
    matched = [kw for kw in keywords_rev if kw in doc.page_content.lower()]
    if len(matched) >= 3:
        base_relevant = doc.page_content[:80]
        break

if base_relevant:
    found_in_hybrid = any(base_relevant in d.page_content for d in docs_hyb)
    print(f"\n⚠️ Doc relevante del baseline {'SÍ' if found_in_hybrid else 'NO'} está en Hybrid v2")


🟢 BASELINE top-5:
  Doc 1: 1 kw → ['contract'] | Revenue and costs are generally directly attributed to our segments. However, due to the integrated ...
  Doc 2: 1 kw → ['contract'] | segment consists of products and services that put customers at the center of the experience with ou...
  Doc 3: 1 kw → ['contract'] | r segments. However, due to the integrated structure of our business, certain revenue recognized and...
  Doc 4: 1 kw → ['contract'] | h our technology. This segment primarily comprises:

 

•

Windows, including Windows original equip...
  Doc 5: 2 kw → ['revenue recognition', 'contract'] | that we experience may differ materially from our estimates.
Note 2: 
Accounting Policies
Revenue Re...

------------------------------------------------------------


NameError: name 'search_hybrid_v2' is not defined

In [24]:
query_rev = "How does the company recognize revenue across different product segments?"

# Buscar los keywords REALES del eval_queries_hard para esta query
for item in eval_queries_hard:
    if "revenue" in item["query"].lower():
        print(f"Query: {item['query']}")
        print(f"Keywords: {item['keywords']}")
        
        # Verificar contra el Doc 5 del baseline
        docs = search_baseline(query_rev, k=5)
        for i, doc in enumerate(docs, 1):
            matched = [kw for kw in item["keywords"] if kw in doc.page_content.lower()]
            print(f"\n  Doc {i}: {len(matched)} matches → {matched}")
            print(f"          {doc.page_content[:200]}")
        break


Query: How does the company recognize revenue across different product segments?
Keywords: ['revenue recognition', 'asc 606', 'performance obligation', 'contract', 'segment']

  Doc 1: 2 matches → ['contract', 'segment']
          Revenue and costs are generally directly attributed to our segments. However, due to the integrated structure of our business, certain revenue recognized and costs incurred by one segment may benefit 

  Doc 2: 2 matches → ['contract', 'segment']
          segment consists of products and services that put customers at the center of the experience with our technology. This segment primarily comprises: 

 

•

Windows, including Windows OEM licensing and

  Doc 3: 2 matches → ['contract', 'segment']
          r segments. However, due to the integrated structure of our business, certain revenue recognized and costs incurred by one segment may benefit other segments. Revenue from certain contracts is allocat

  Doc 4: 2 matches → ['contract', 'segment']
        

In [25]:
def search_hybrid_v3(query, k=5):
    """Hybrid: semántica domina, BM25 solo complementa."""
    from rank_bm25 import BM25Okapi
    
    # Semántica: top k direct (GARANTIZADOS)
    sem_top = vectorstore.similarity_search(query, k=k)
    
    # BM25: top k
    corpus_tokens = [doc.page_content.lower().split() for doc in all_docs]
    bm25 = BM25Okapi(corpus_tokens)
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_top_idx = bm25_scores.argsort()[::-1][:k]
    bm25_top = [all_docs[i] for i in bm25_top_idx]
    
    # Merge: empezar con semánticos, agregar BM25 no duplicados
    final = list(sem_top)
    seen = set(doc.page_content for doc in final)
    
    for doc in bm25_top:
        if len(final) >= k:
            break
        if doc.page_content not in seen:
            # Reemplazar el último semántico por el mejor BM25 no visto
            final.insert(len(final) - 1, doc)
            seen.add(doc.page_content)
    
    return final[:k]

def search_reranking_v3(query, k=5):
    """Re-ranking sobre pool amplio."""
    # Pool: top-15 semánticos + top-15 BM25
    from rank_bm25 import BM25Okapi
    
    sem_pool = vectorstore.similarity_search(query, k=15)
    
    corpus_tokens = [doc.page_content.lower().split() for doc in all_docs]
    bm25 = BM25Okapi(corpus_tokens)
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_top_idx = bm25_scores.argsort()[::-1][:15]
    bm25_pool = [all_docs[i] for i in bm25_top_idx]
    
    # Deduplicar
    seen = set()
    candidates = []
    for doc in sem_pool + bm25_pool:
        if doc.page_content not in seen:
            candidates.append(doc)
            seen.add(doc.page_content)
    
    # Re-rank con Cross-Encoder
    pairs = [[query, doc.page_content] for doc in candidates]
    scores = cross_encoder.predict(pairs)
    top_indices = scores.argsort()[::-1][:k]
    return [candidates[i] for i in top_indices]

# Evaluar v3
techniques_v3 = {
    "1. Baseline (Vector)":      search_baseline,
    "2. Metadata (10-K)":        search_metadata,
    "3. Hybrid v3 (Sem+BM25)":   search_hybrid_v3,
    "4. Re-ranking v3 (CE)":     search_reranking_v3
}

results_table = []
for tech_name, search_fn in techniques_v3.items():
    hits = 0
    mrr_sum = 0.0
    for eval_item in eval_queries_hard:
        query = eval_item["query"]
        keywords = eval_item["keywords"]
        try:
            docs = search_fn(query, k=5)
        except Exception as e:
            print(f"⚠️ {tech_name}: {e}")
            docs = []
        for rank, doc in enumerate(docs, start=1):
            if is_relevant(doc.page_content, keywords, min_matches=3):
                hits += 1
                mrr_sum += 1.0 / rank
                break
    n = len(eval_queries_hard)
    results_table.append({
        "Technique": tech_name,
        "Hit Rate": f"{hits/n:.2%}",
        "MRR": f"{mrr_sum/n:.4f}",
        "Hits": f"{hits}/{n}"
    })
    print(f"✅ {tech_name}: HR={hits/n:.2%}, MRR={mrr_sum/n:.4f}")

print("\n" + "="*60)
print("📊 ABLATION TABLE v3 - RAG Pipeline SEC Edgar")
print("="*60)
df = pd.DataFrame(results_table)
print(df.to_string(index=False))


✅ 1. Baseline (Vector): HR=87.50%, MRR=0.5979
✅ 2. Metadata (10-K): HR=87.50%, MRR=0.7188
✅ 3. Hybrid v3 (Sem+BM25): HR=87.50%, MRR=0.5979
✅ 4. Re-ranking v3 (CE): HR=87.50%, MRR=0.5833

📊 ABLATION TABLE v3 - RAG Pipeline SEC Edgar
              Technique Hit Rate    MRR Hits
   1. Baseline (Vector)   87.50% 0.5979  7/8
     2. Metadata (10-K)   87.50% 0.7188  7/8
3. Hybrid v3 (Sem+BM25)   87.50% 0.5979  7/8
  4. Re-ranking v3 (CE)   87.50% 0.5833  7/8


In [26]:
def evaluate_retrieval(results, keywords, min_matches=3):
    """Evalúa si al menos un doc en results es relevante (3+ keyword matches)."""
    for rank, doc in enumerate(results, 1):
        text = doc.page_content.lower()
        matched = [kw for kw in keywords if kw in text]
        if len(matched) >= min_matches:
            return 1, 1.0 / rank  # hit=1, reciprocal rank
    return 0, 0.0  # no hit


In [29]:
from langchain_core.documents import Document

# Extraer todos los documentos de ChromaDB
all_data = collection.get(include=["documents", "metadatas"])

enriched_docs = []
for text, meta in zip(all_data["documents"], all_data["metadatas"]):
    enriched_docs.append(Document(page_content=text, metadata=meta))

print(f"✅ enriched_docs recargados: {len(enriched_docs)} documentos")



NameError: name 'collection' is not defined

In [30]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")  # ajusta la ruta si es diferente
collection = client.get_collection("langchain")  
# ── Hybrid v4: Metadata + Semántica + BM25 + Re-ranking ──

def search_hybrid_v4(query, k=5, filing_type="10-K"):
    """Combina filtrado metadata + semántico + BM25 + CE reranking"""
    
    # 1. Semántica CON filtro metadata (lo mejor de ambos mundos)
    sem_results = collection.query(
        query_texts=[query],
        n_results=15,
        where={"filing_type": filing_type}
    )
    sem_docs = []
    for doc_text, meta in zip(sem_results["documents"][0], sem_results["metadatas"][0]):
        sem_docs.append(Document(page_content=doc_text, metadata=meta))
    
    # 2. BM25 sobre el mismo subset filtrado (solo 10-K docs)
    filtered_docs = [d for d in enriched_docs if d.metadata.get("filing_type") == filing_type]
    filtered_corpus = [d.page_content.lower().split() for d in filtered_docs]
    bm25_filtered = BM25Okapi(filtered_corpus)
    
    tokenized_query = query.lower().split()
    bm25_scores = bm25_filtered.get_scores(tokenized_query)
    top_bm25_idx = bm25_scores.argsort()[-10:][::-1]
    bm25_docs = [filtered_docs[i] for i in top_bm25_idx]
    
    # 3. Merge con garantía semántica
    seen = set()
    merged = []
    for doc in sem_docs:
        key = doc.page_content[:200]
        if key not in seen:
            seen.add(key)
            merged.append(doc)
    for doc in bm25_docs:
        key = doc.page_content[:200]
        if key not in seen and len(merged) < 30:
            seen.add(key)
            merged.append(doc)
    
    # 4. Re-ranking con Cross-Encoder
    pairs = [[query, doc.page_content] for doc in merged]
    ce_scores = cross_encoder.predict(pairs)
    
    ranked_idx = ce_scores.argsort()[::-1][:k]
    return [merged[i] for i in ranked_idx]


# ── Evaluación completa v4 ──

techniques_v4 = {
    "1. Baseline (Vector)":        lambda q: search_baseline(q, k=5),
    "2. Metadata (10-K)":          lambda q: search_metadata(q, k=5),
    "3. Hybrid v3 (Sem+BM25)":     lambda q: search_hybrid_v3(q, k=5),
    "4. Hybrid v4 (Meta+Sem+BM25+CE)": lambda q: search_hybrid_v4(q, k=5),
}

print("=" * 60)
print("📊 ABLATION TABLE v4 - RAG Pipeline SEC Edgar")
print("=" * 60)

all_results_v4 = {}
for name, search_fn in techniques_v4.items():
    hits, recips = 0, 0.0
    for item in eval_queries_hard:
        results = search_fn(item["query"])
        hit, rr = evaluate_retrieval(results, item["keywords"])
        hits += hit
        recips += rr
    n = len(eval_queries_hard)
    hr = hits / n
    mrr = recips / n
    all_results_v4[name] = {"hr": hr, "mrr": mrr, "hits": hits, "total": n}
    print(f"✅ {name}: HR={hr*100:.2f}%, MRR={mrr:.4f}")

print("\n" + "=" * 60)
print(f"{'Technique':>40s} {'Hit Rate':>9s} {'MRR':>7s} {'Hits':>5s}")
print("=" * 60)
for name, r in all_results_v4.items():
    print(f"{name:>40s} {r['hr']*100:>8.2f}% {r['mrr']:>7.4f} {r['hits']:>2d}/{r['total']}")


📊 ABLATION TABLE v4 - RAG Pipeline SEC Edgar
✅ 1. Baseline (Vector): HR=87.50%, MRR=0.5979
✅ 2. Metadata (10-K): HR=87.50%, MRR=0.7188
✅ 3. Hybrid v3 (Sem+BM25): HR=87.50%, MRR=0.5979


NameError: name 'enriched_docs' is not defined